## Week 8 Assignment : E-Commerce Order Analytics System

### Name: Soham Sunilrao Deshmukh

### Phase 1 - Data Generation

The raw datasets were generated using the Python script:

`src/data_generation.py`

The generated files are stored in:

`data/raw/`

Files:
- customers.csv
- products.csv
- orders.csv
- order_items.csv

In [291]:
import pandas as pd
import sqlite3
import os
from datetime import datetime, timedelta

In [292]:
# paths
RAW_DIR = "../data/raw"
CLEAN_DIR = "../data/cleaned"
OUTPUT_DIR = "../output"

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

DB_PATH = "../output/ecommerce.db"

In [293]:
# Load raw CSV files

orders = pd.read_csv(RAW_DIR + "/orders.csv")
order_items = pd.read_csv(RAW_DIR + "/order_items.csv")
products = pd.read_csv(RAW_DIR + "/products.csv")
customers = pd.read_csv(RAW_DIR + "/customers.csv")

print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)

Orders: (1000, 5)
Order Items: (3000, 6)
Products: (640, 5)
Customers: (800, 5)


In [294]:
# Load raw CSV files

orders = pd.read_csv(RAW_DIR + "/orders.csv")
order_items = pd.read_csv(RAW_DIR + "/order_items.csv")
products = pd.read_csv(RAW_DIR + "/products.csv")
customers = pd.read_csv(RAW_DIR + "/customers.csv")

print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)

Orders: (1000, 5)
Order Items: (3000, 6)
Products: (640, 5)
Customers: (800, 5)


In [295]:
# Check number of records

print("Orders:", len(orders))
print("Order Items:", len(order_items))
print("Products:", len(products))
print("Customers:", len(customers))

Orders: 1000
Order Items: 3000
Products: 640
Customers: 800


### 2.2 Check Missing Values

First, we check the raw datasets for missing values.

In [296]:
print("Missing values in orders:")
print(orders.isnull().sum())

print("\nMissing values in order_items:")
print(order_items.isnull().sum())

print("\nMissing values in products:")
print(products.isnull().sum())

print("\nMissing values in customers:")
print(customers.isnull().sum())

Missing values in orders:
order_id        0
customer_id    50
order_date      0
status          0
region_code     0
dtype: int64

Missing values in order_items:
item_id             0
order_id            0
product_id          0
quantity            0
unit_price          0
discount_percent    0
dtype: int64

Missing values in products:
product_id      0
product_name    0
category        0
subcategory     0
cost_price      0
dtype: int64

Missing values in customers:
customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
dtype: int64


In [297]:
# Missing customer IDs

missing_customer_ids = orders[
    orders["customer_id"].isna() |
    (orders["customer_id"].astype(str).str.strip() == "") |
    (orders["customer_id"].astype(str).str.upper() == "NULL")
]

print("Orders with missing customer_id:",
      len(missing_customer_ids))

Orders with missing customer_id: 50


In [298]:
# Negative quantities

negative_quantity = order_items[
    order_items["quantity"] < 0
]

print("Order items with negative quantity:",
      len(negative_quantity))

Order items with negative quantity: 90


In [299]:
# Check wrong date format

wrong_date_format = orders[
    orders["order_date"].astype(str).str.match(
        r"^\d{2}-\d{2}-\d{4}"
    )
]

print("Orders with DD-MM-YYYY format:",
      len(wrong_date_format))

Orders with DD-MM-YYYY format: 10


In [300]:
# Check product names with extra spaces

extra_space_products = products[
    products["product_name"] !=
    products["product_name"].str.strip()
]

print("Products with extra spaces:",
      len(extra_space_products))

Products with extra spaces: 32


### 2.4 clean_orders()

Requirements:
- Fix incorrect date formats
- Handle NULL customer IDs

The assignment specifically asks for the `clean_orders()` function. 

In [301]:
def clean_orders(df):

    df = df.copy()

    # Handle NULL and empty customer IDs
    df["customer_id"] = df["customer_id"].replace(
        ["NULL", "", "null"],
        pd.NA
    )

    # Convert both supported date formats
    df["order_date"] = pd.to_datetime(
        df["order_date"],
        dayfirst=True,
        errors="coerce"
    )

    return df

In [302]:
orders_cleaned = clean_orders(orders)

print("Orders cleaned successfully.")

display(orders_cleaned.head())

Orders cleaned successfully.


,order_id,customer_id,order_date,status,region_code
0,O00001,NaN,2026-09-05 12:35:18,DELIVERED,WEST
1,O00002,NaN,2025-02-06 20:19:43,RETURNED,CENTRAL
2,O00003,NaN,NaT,DELIVERED,SOUTH
3,O00004,NaN,2026-02-06 17:05:10,RETURNED,SOUTH
4,O00005,NaN,NaT,DELIVERED,EAST


In [303]:
print("Invalid dates after cleaning:")

print(
    orders_cleaned[
        orders_cleaned["order_date"].isna()
    ]
)

Invalid dates after cleaning:
    order_id customer_id order_date     status region_code
2     O00003         NaN        NaT  DELIVERED       SOUTH
4     O00005         NaN        NaT  DELIVERED        EAST
6     O00007         NaN        NaT  CANCELLED       NORTH
7     O00008         NaN        NaT  CANCELLED        WEST
9     O00010         NaN        NaT     PLACED        WEST
..       ...         ...        ...        ...         ...
984   O00985    CUST0742        NaT  CANCELLED       SOUTH
985   O00986    CUST0188        NaT    SHIPPED        EAST
986   O00987    CUST0109        NaT  DELIVERED       NORTH
988   O00989    CUST0387        NaT   RETURNED     CENTRAL
989   O00990    CUST0450        NaT   RETURNED       NORTH

[571 rows x 5 columns]


### 2.5 clean_products()

Requirements:
- Remove extra spaces
- Normalize product names
- Convert names to title case

In [304]:
def clean_products(df):

    df = df.copy()

    df["product_name"] = (
        df["product_name"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    return df

In [305]:
products_cleaned = clean_products(products)

print("Products cleaned successfully.")

display(products_cleaned.head())

Products cleaned successfully.


,product_id,product_name,category,subcategory,cost_price
0,P0001,Smartphone 1,Electronics,Gadgets,28271.31
1,P0002,Smartphone 2,Electronics,Gadgets,1566.60
2,P0003,Smartphone 3,Electronics,Gadgets,27727.75
3,P0004,Smartphone 4,Electronics,Gadgets,13023.03
4,P0005,Smartphone 5,Electronics,Gadgets,21746.70


### 2.6 validate_emails()

A valid email should contain:
- an `@`
- a domain
- a `.`

In [306]:
def validate_emails(df):

    invalid_customer_ids = []

    for index, row in df.iterrows():

        email = str(row["email"])

        if (
            "@" not in email
            or "." not in email.split("@")[-1]
        ):
            invalid_customer_ids.append(
                row["customer_id"]
            )

    return invalid_customer_ids

In [307]:
invalid_customer_ids = validate_emails(customers)

print("Invalid customer IDs:")
print(invalid_customer_ids)

print("\nTotal invalid emails:",
      len(invalid_customer_ids))

Invalid customer IDs:
['CUST0050', 'CUST0100', 'CUST0150', 'CUST0200', 'CUST0250', 'CUST0300', 'CUST0350', 'CUST0400', 'CUST0450', 'CUST0500', 'CUST0550', 'CUST0600', 'CUST0650', 'CUST0700', 'CUST0750', 'CUST0800']

Total invalid emails: 16


In [308]:
def check_referential_integrity(order_items_df, orders_df):

    valid_order_ids = set(
        orders_df["order_id"]
    )

    invalid_items = order_items_df[
        ~order_items_df["order_id"].isin(
            valid_order_ids
        )
    ]

    return invalid_items

In [309]:
invalid_order_items = check_referential_integrity(
    order_items,
    orders_cleaned
)

print(
    "Order items with invalid order_id:",
    len(invalid_order_items)
)

display(invalid_order_items.head())

Order items with invalid order_id: 20


,item_id,order_id,product_id,quantity,unit_price,discount_percent
2980,OI002981,O99902981,P0496,3,24728.14,3.77
2981,OI002982,O99902982,P0280,2,33153.40,9.22
2982,OI002983,O99902983,P0389,5,23976.54,3.31
2983,OI002984,O99902984,P0346,4,16170.10,18.14
2984,OI002985,O99902985,P0366,2,6721.85,17.24


In [310]:
# Check discounts greater than 100

invalid_discount = order_items[
    order_items["discount_percent"] > 100
]

print(
    "Discounts greater than 100:",
    len(invalid_discount)
)

Discounts greater than 100: 20


In [311]:
# Check zero quantities

zero_quantity = order_items[
    order_items["quantity"] == 0
]

print(
    "Order items with quantity = 0:",
    len(zero_quantity)
)

Order items with quantity = 0: 20


In [312]:
# Convert date to required standard format

orders_cleaned["order_date"] = (
    orders_cleaned["order_date"]
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

In [313]:
# Save cleaned CSV files

customers.to_csv(
    CLEAN_DIR + "/customers_cleaned.csv",
    index=False
)

products_cleaned.to_csv(
    CLEAN_DIR + "/products_cleaned.csv",
    index=False
)

orders_cleaned.to_csv(
    CLEAN_DIR + "/orders_cleaned.csv",
    index=False
)

order_items.to_csv(
    CLEAN_DIR + "/order_items_cleaned.csv",
    index=False
)

print("All cleaned CSV files saved.")

All cleaned CSV files saved.


In [314]:
issues = []

# Missing customer IDs

for order_id in missing_customer_ids["order_id"]:
    issues.append([
        "orders",
        order_id,
        "Missing customer_id"
    ])


# Negative quantities

for item_id in negative_quantity["item_id"]:
    issues.append([
        "order_items",
        item_id,
        "Negative quantity / return"
    ])


# Invalid emails

for customer_id in invalid_customer_ids:
    issues.append([
        "customers",
        customer_id,
        "Invalid email"
    ])


# Invalid order references

for item_id in invalid_order_items["item_id"]:
    issues.append([
        "order_items",
        item_id,
        "Invalid order_id"
    ])

In [315]:
# Create issue report

issue_report = pd.DataFrame(
    issues,
    columns=[
        "table_name",
        "record_id",
        "issue"
    ]
)

display(issue_report.head())

# Save issue report in WEEK-8/output/

issue_report.to_csv(
    OUTPUT_DIR + "/issue_report.csv",
    index=False
)

print("Issue report saved to WEEK-8/output/issue_report.csv")

,table_name,record_id,issue
0,orders,O00001,Missing customer_id
1,orders,O00002,Missing customer_id
2,orders,O00003,Missing customer_id
3,orders,O00004,Missing customer_id
4,orders,O00005,Missing customer_id


Issue report saved to WEEK-8/output/issue_report.csv


In [316]:
print("========== Part 2 SUMMARY Report ==========")

print(
    "Missing customer IDs:",
    len(missing_customer_ids)
)

print(
    "Negative quantities:",
    len(negative_quantity)
)

print(
    "Invalid emails:",
    len(invalid_customer_ids)
)

print(
    "Invalid order references:",
    len(invalid_order_items)
)

print(
    "Total reported issues:",
    len(issue_report)
)

========== Part 2 SUMMARY Report ==========
Missing customer IDs: 50
Negative quantities: 90
Invalid emails: 16
Invalid order references: 20
Total reported issues: 176


## Phase 3 - SQL Analysis

In this phase, the cleaned data is loaded into a SQLite database.

Tables:
- customers
- products
- orders
- order_items

We will then perform the required SQL analysis queries.

In [317]:
# Connect to SQLite database

conn = sqlite3.connect(DB_PATH)

print("SQLite database connected successfully.")

SQLite database connected successfully.


In [318]:
# Create customers table

conn.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT,
    email TEXT,
    registration_date TEXT,
    customer_type TEXT
)
""")

# Create products table

conn.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    subcategory TEXT,
    cost_price REAL
)
""")

# Create orders table

conn.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_date TEXT,
    status TEXT,
    region_code TEXT
)
""")

# Create order_items table

conn.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    item_id TEXT PRIMARY KEY,
    order_id TEXT,
    product_id TEXT,
    quantity INTEGER,
    unit_price REAL,
    discount_percent REAL
)
""")

conn.commit()

print("All tables created successfully.")

All tables created successfully.


In [319]:
# Read cleaned CSV files

customers_cleaned = pd.read_csv(
    CLEAN_DIR + "/customers_cleaned.csv"
)

products_cleaned = pd.read_csv(
    CLEAN_DIR + "/products_cleaned.csv"
)

orders_cleaned = pd.read_csv(
    CLEAN_DIR + "/orders_cleaned.csv"
)

order_items_cleaned = pd.read_csv(
    CLEAN_DIR + "/order_items_cleaned.csv"
)

In [320]:
# Load data into SQLite

customers_cleaned.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

products_cleaned.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

orders_cleaned.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

order_items_cleaned.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("Cleaned data loaded into SQLite successfully.")

Cleaned data loaded into SQLite successfully.


In [321]:
# Check number of records in each table

print("Customers:")
display(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM customers",
        conn
    )
)

print("Products:")
display(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM products",
        conn
    )
)

print("Orders:")
display(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM orders",
        conn
    )
)

print("Order Items:")
display(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM order_items",
        conn
    )
)

Customers:


,total
0,800


Products:


,total
0,640


Orders:


,total
0,1000


Order Items:


,total
0,3000


### SQL 1 - Total Revenue per Category

Calculate total revenue for every product category and sort the result from highest to lowest revenue.

formula for revenue calculation : quantity × unit_price × (1 - discount_percent / 100)

In [322]:
query1 = """
SELECT
    p.category,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ),
        2
    ) AS total_revenue

FROM order_items oi

JOIN products p
    ON oi.product_id = p.product_id

GROUP BY
    p.category

ORDER BY
    total_revenue DESC
"""

result1 = pd.read_sql(query1, conn)

display(result1)

,category,total_revenue
0,Home,38626797.22
1,Books,38534850.59
2,Clothing,34087400.87
3,Electronics,32748482.16


### SQL 2 - Top 10 Customers by Total Order Value

Find the top 10 customers based on their total order value.

In [323]:
query2 = """
SELECT
    o.customer_id,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ),
        2
    ) AS total_order_value

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.customer_id IS NOT NULL

GROUP BY
    o.customer_id

ORDER BY
    total_order_value DESC

LIMIT 10
"""

result2 = pd.read_sql(query2, conn)

display(result2)

,customer_id,total_order_value
0,CUST0493,1130096.19
1,CUST0759,971443.25
2,CUST0470,920614.73
3,CUST0303,909454.15
4,CUST0129,900619.65
5,CUST0355,793319.35
6,CUST0444,777645.55
7,CUST0044,766593.06
8,CUST0539,757852.30
9,CUST0763,744206.10


### SQL 3 - Month-wise Order Count

Calculate the number of orders for each month and display the latest 12 months.

In [324]:
query3 = """
SELECT
    strftime('%Y-%m', order_date) AS month,
    COUNT(*) AS order_count

FROM orders

GROUP BY
    month

ORDER BY
    month DESC

LIMIT 12
"""

result3 = pd.read_sql(query3, conn)

display(result3)

,month,order_count
0,2026-12,8
1,2026-11,13
2,2026-10,13
3,2026-09,19
4,2026-08,19
5,2026-07,13
6,2026-06,11
7,2026-05,16
8,2026-04,18
9,2026-03,16


### SQL 4 - Customers Who Never Had a Delivered Order

Find customers who placed orders but never had an order with status DELIVERED.

In [325]:
query4 = """
SELECT DISTINCT
    o.customer_id

FROM orders o

WHERE o.customer_id IS NOT NULL

AND o.customer_id NOT IN (

    SELECT customer_id

    FROM orders

    WHERE status = 'DELIVERED'
)

ORDER BY
    o.customer_id
"""

result4 = pd.read_sql(query4, conn)

display(result4)

,customer_id


### SQL 5 - Products with More Returns Than Purchases

Identify products where the total returned quantity is greater than the total purchased quantity.

In [326]:
query5 = """
SELECT
    p.product_id,
    p.product_name,

    SUM(
        CASE
            WHEN oi.quantity > 0
            THEN oi.quantity
            ELSE 0
        END
    ) AS purchases,

    SUM(
        CASE
            WHEN oi.quantity < 0
            THEN ABS(oi.quantity)
            ELSE 0
        END
    ) AS returns

FROM products p

JOIN order_items oi
    ON p.product_id = oi.product_id

GROUP BY
    p.product_id,
    p.product_name

HAVING
    returns > purchases

ORDER BY
    returns DESC
"""

result5 = pd.read_sql(query5, conn)

display(result5)

,product_id,product_name,purchases,returns
0,P0001,Smartphone 1,0,176


### SQL 6 - Return Rate per Category

Calculate the percentage of returned quantity for each category.

In [327]:
query6 = """
SELECT
    p.category,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN oi.quantity < 0
                THEN ABS(oi.quantity)
                ELSE 0
            END
        )
        /
        NULLIF(SUM(ABS(oi.quantity)), 0),
        2
    ) AS return_rate

FROM products p

JOIN order_items oi
    ON p.product_id = oi.product_id

GROUP BY
    p.category

ORDER BY
    return_rate DESC
"""

result6 = pd.read_sql(query6, conn)

display(result6)

,category,return_rate
0,Electronics,7.34
1,Home,0.00
2,Clothing,0.00
3,Books,0.00


### SQL 7 - Running Revenue Total per Region

Calculate:
- region_code
- order_date
- daily_revenue
- running_total

The running total is calculated separately for each region.

In [328]:
query7 = """
WITH daily_revenue AS (

    SELECT
        o.region_code,
        DATE(o.order_date) AS order_date,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS daily_revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    GROUP BY
        o.region_code,
        DATE(o.order_date)
)

SELECT
    region_code,
    order_date,

    ROUND(daily_revenue, 2) AS daily_revenue,

    ROUND(
        SUM(daily_revenue) OVER (
            PARTITION BY region_code
            ORDER BY order_date
        ),
        2
    ) AS running_total

FROM daily_revenue

ORDER BY
    region_code,
    order_date
"""

result7 = pd.read_sql(query7, conn)

display(result7.head(20))

,region_code,order_date,daily_revenue,running_total
0,CENTRAL,NaN,17015199.25,17015199.25
1,CENTRAL,2025-01-03,76294.37,17091493.62
2,CENTRAL,2025-02-05,157421.42,17248915.04
3,CENTRAL,2025-02-06,617118.71,17866033.76
4,CENTRAL,2025-02-09,16243.72,17882277.47
5,CENTRAL,2025-02-12,59442.60,17941720.07
6,CENTRAL,2025-03-06,316994.26,18258714.34
7,CENTRAL,2025-03-09,131972.37,18390686.71
8,CENTRAL,2025-03-11,166985.86,18557672.57
9,CENTRAL,2025-04-06,56418.68,18614091.25


### SQL 8 - Product Ranking Within Category

Rank products by revenue within each category using DENSE_RANK.

Products having the same revenue should receive the same rank.

In [329]:
query8 = """
WITH product_revenue AS (

    SELECT
        p.category,
        p.product_id,
        p.product_name,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS total_revenue

    FROM products p

    JOIN order_items oi
        ON p.product_id = oi.product_id

    GROUP BY
        p.category,
        p.product_id,
        p.product_name
)

SELECT
    category,
    product_name,

    ROUND(total_revenue, 2) AS total_revenue,

    DENSE_RANK() OVER (
        PARTITION BY category
        ORDER BY total_revenue DESC
    ) AS rank_in_category

FROM product_revenue

ORDER BY
    category,
    rank_in_category
"""

result8 = pd.read_sql(query8, conn)

display(result8.head(20))

,category,product_name,total_revenue,rank_in_category
0,Books,Python Guide 20,892437.84,1
1,Books,Algorithms 6,823851.21,2
2,Books,Database Design 19,820174.12,3
3,Books,Algorithms 2,721931.74,4
4,Books,Ai Fundamentals 12,607741.99,5
5,Books,Database Design 15,607074.82,6
6,Books,Algorithms 11,599877.64,7
7,Books,Spark Guide 1,590947.07,8
8,Books,Database Design 4,557571.56,9
9,Books,Data Science 1,557016.90,10


### SQL 9 - Consecutive Order Gap

Calculate the number of days between consecutive orders for each customer.

Customers whose average order gap is greater than 30 days are marked as At Risk.

In [330]:
query9 = """
WITH customer_orders AS (

    SELECT
        customer_id,
        order_date,

        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_date

    FROM orders

    WHERE customer_id IS NOT NULL
),

order_gaps AS (

    SELECT
        customer_id,
        order_date,
        previous_order_date,

        ROUND(
            julianday(order_date)
            - julianday(previous_order_date),
            2
        ) AS days_gap

    FROM customer_orders
),

average_gaps AS (

    SELECT
        customer_id,
        AVG(days_gap) AS average_gap

    FROM order_gaps

    WHERE days_gap IS NOT NULL

    GROUP BY customer_id
)

SELECT
    g.customer_id,
    g.order_date,
    g.previous_order_date,
    g.days_gap,

    CASE
        WHEN a.average_gap > 30
        THEN 'At Risk'
        ELSE 'Active'
    END AS customer_status

FROM order_gaps g

JOIN average_gaps a
    ON g.customer_id = a.customer_id

ORDER BY
    g.customer_id,
    g.order_date
"""

result9 = pd.read_sql(query9, conn)

display(result9.head(20))

,customer_id,order_date,previous_order_date,days_gap,customer_status
0,CUST0001,2025-10-01 10:00:00,NaN,NaN,Active
1,CUST0001,2025-10-02 10:00:00,2025-10-01 10:00:00,1.00,Active
2,CUST0001,2025-10-03 10:00:00,2025-10-02 10:00:00,1.00,Active
3,CUST0001,2025-10-04 10:00:00,2025-10-03 10:00:00,1.00,Active
4,CUST0012,NaN,NaN,NaN,At Risk
5,CUST0012,2025-06-03 21:19:16,NaN,NaN,At Risk
6,CUST0012,2025-08-01 06:31:56,2025-06-03 21:19:16,58.38,At Risk
7,CUST0022,NaN,NaN,NaN,At Risk
8,CUST0022,NaN,NaN,NaN,At Risk
9,CUST0022,2025-07-02 12:21:10,NaN,NaN,At Risk


### SQL 10 - Multi-Level CTE

Steps:
1. Calculate monthly revenue for each customer.
2. Categorize customers:
   - High
   - Medium
   - Low
3. Count customers in each category for every month.

In [331]:
query10 = """
WITH monthly_revenue AS (

    SELECT
        o.customer_id,

        strftime(
            '%Y-%m',
            o.order_date
        ) AS month,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.customer_id IS NOT NULL

    GROUP BY
        o.customer_id,
        month
),

customer_category AS (

    SELECT
        customer_id,
        month,
        revenue,

        CASE
            WHEN revenue > 10000
                THEN 'High'

            WHEN revenue >= 5000
                THEN 'Medium'

            ELSE 'Low'
        END AS revenue_category

    FROM monthly_revenue
)

SELECT
    month,
    revenue_category,
    COUNT(*) AS customer_count

FROM customer_category

GROUP BY
    month,
    revenue_category

ORDER BY
    month,
    revenue_category
"""

result10 = pd.read_sql(query10, conn)

display(result10)

,month,revenue_category,customer_count
0,NaN,High,356
1,NaN,Low,13
2,NaN,Medium,3
3,2025-01,High,13
4,2025-02,High,16
5,2025-03,High,18
6,2025-04,High,18
7,2025-05,High,20
8,2025-05,Low,3
9,2025-05,Medium,2


### SQL 11 - Customer Lifetime Value Quartiles

Divide customers into four groups using NTILE(4).

Labels:
- Platinum
- Gold
- Silver
- Bronze

In [332]:
query11 = """
WITH customer_value AS (

    SELECT
        o.customer_id,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS total_value

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.customer_id IS NOT NULL

    GROUP BY
        o.customer_id
),

customer_quartiles AS (

    SELECT
        customer_id,
        total_value,

        NTILE(4) OVER (
            ORDER BY total_value DESC
        ) AS quartile

    FROM customer_value
)

SELECT
    customer_id,

    ROUND(total_value, 2) AS total_value,

    quartile,

    CASE
        WHEN quartile = 1 THEN 'Platinum'
        WHEN quartile = 2 THEN 'Gold'
        WHEN quartile = 3 THEN 'Silver'
        WHEN quartile = 4 THEN 'Bronze'
    END AS quartile_label

FROM customer_quartiles

ORDER BY
    quartile,
    total_value DESC
"""

result11 = pd.read_sql(query11, conn)

display(result11)

,customer_id,total_value,quartile,quartile_label
0,CUST0493,1130096.19,1,Platinum
1,CUST0759,971443.25,1,Platinum
2,CUST0470,920614.73,1,Platinum
3,CUST0303,909454.15,1,Platinum
4,CUST0129,900619.65,1,Platinum
...,...,...,...,...
527,CUST0191,-45694.39,4,Bronze
528,CUST0104,-62485.32,4,Bronze
529,CUST0496,-64676.99,4,Bronze
530,CUST0645,-76650.42,4,Bronze


### SQL 12 - Year-over-Year Revenue

Compare monthly revenue with the same month of the previous year.

If previous-year data does not exist, the growth value should be NULL.

In [333]:
query12 = """
WITH monthly_revenue AS (

    SELECT
        strftime('%Y', o.order_date) AS year,
        strftime('%m', o.order_date) AS month,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    GROUP BY
        year,
        month
)

SELECT
    current.year,
    current.month,

    ROUND(
        current.revenue,
        2
    ) AS revenue,

    ROUND(
        previous.revenue,
        2
    ) AS previous_year_revenue,

    CASE
        WHEN previous.revenue IS NULL
            THEN NULL

        WHEN previous.revenue = 0
            THEN NULL

        ELSE ROUND(
            (
                (
                    current.revenue
                    - previous.revenue
                )
                / previous.revenue
            ) * 100,
            2
        )
    END AS yoy_growth_percent

FROM monthly_revenue current

LEFT JOIN monthly_revenue previous

    ON previous.month = current.month

    AND CAST(previous.year AS INTEGER)
        =
        CAST(current.year AS INTEGER) - 1

ORDER BY
    current.year,
    current.month
"""

result12 = pd.read_sql(query12, conn)

display(result12)

,year,month,revenue,previous_year_revenue,yoy_growth_percent
0,NaN,NaN,81876042.52,NaN,NaN
1,2025,01,2922446.38,NaN,NaN
2,2025,02,3110242.75,NaN,NaN
3,2025,03,4031361.10,NaN,NaN
4,2025,04,3739623.08,NaN,NaN
5,2025,05,4222285.19,NaN,NaN
6,2025,06,3477357.86,NaN,NaN
7,2025,07,2908648.60,NaN,NaN
8,2025,08,2302304.06,NaN,NaN
9,2025,09,3103738.27,NaN,NaN


### SQL 13 - First and Last Purchased Category

For each customer, find:

- first purchased category
- most recent purchased category
- whether the customer changed category

In [334]:
query13 = """
WITH customer_categories AS (

    SELECT
        o.customer_id,
        o.order_date,
        p.category,

        FIRST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        ) AS first_category,

        LAST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND UNBOUNDED FOLLOWING
        ) AS last_category

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE o.customer_id IS NOT NULL
)

SELECT DISTINCT
    customer_id,
    first_category,
    last_category,

    CASE
        WHEN first_category = last_category
            THEN 'No'
        ELSE 'Yes'
    END AS category_shift

FROM customer_categories

ORDER BY customer_id
"""

result13 = pd.read_sql(query13, conn)

display(result13)

,customer_id,first_category,last_category,category_shift
0,CUST0001,Clothing,Home,Yes
1,CUST0002,Clothing,Home,Yes
2,CUST0003,Clothing,Home,Yes
3,CUST0004,Clothing,Clothing,No
4,CUST0005,Clothing,Home,Yes
...,...,...,...,...
527,CUST0795,Electronics,Books,Yes
528,CUST0797,Clothing,Books,Yes
529,CUST0798,Electronics,Books,Yes
530,CUST0799,Electronics,Books,Yes


### SQL 14 - Cumulative Revenue Distribution

Calculate:

- customer revenue
- cumulative revenue
- cumulative percentage of total revenue

In [335]:
query14 = """
WITH customer_revenue AS (

    SELECT
        o.customer_id,

        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.customer_id IS NOT NULL

    GROUP BY
        o.customer_id
),

cumulative_revenue AS (

    SELECT
        customer_id,
        revenue,

        SUM(revenue) OVER (
            ORDER BY revenue DESC
        ) AS cumulative_revenue,

        SUM(revenue) OVER () AS total_revenue

    FROM customer_revenue
)

SELECT
    customer_id,

    ROUND(
        revenue,
        2
    ) AS revenue,

    ROUND(
        cumulative_revenue,
        2
    ) AS cumulative_revenue,

    ROUND(
        cumulative_revenue
        * 100.0
        / total_revenue,
        2
    ) AS cumulative_percent

FROM cumulative_revenue

ORDER BY
    revenue DESC
"""

result14 = pd.read_sql(query14, conn)

display(result14)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,CUST0493,1130096.19,1.130096e+06,0.83
1,CUST0759,971443.25,2.101539e+06,1.55
2,CUST0470,920614.73,3.022154e+06,2.22
3,CUST0303,909454.15,3.931608e+06,2.89
4,CUST0129,900619.65,4.832228e+06,3.56
...,...,...,...,...
527,CUST0191,-45694.39,1.362589e+08,100.27
528,CUST0104,-62485.32,1.361964e+08,100.23
529,CUST0496,-64676.99,1.361317e+08,100.18
530,CUST0645,-76650.42,1.360551e+08,100.12


### SQL 15 - Cohort Retention Analysis

Customers are grouped based on their registration month.

Calculate retention for:
- Month 0
- Month 1
- Month 2
- Month 3

In [336]:
query15 = """
WITH customer_cohort AS (

    SELECT
        customer_id,

        strftime(
            '%Y-%m',
            registration_date
        ) AS cohort_month

    FROM customers
),

customer_orders AS (

    SELECT DISTINCT
        customer_id,

        strftime(
            '%Y-%m',
            order_date
        ) AS order_month

    FROM orders

    WHERE customer_id IS NOT NULL
),

cohort_activity AS (

    SELECT
        c.cohort_month,
        o.customer_id,
        o.order_month,

        (
            (
                CAST(
                    substr(o.order_month, 1, 4)
                    AS INTEGER
                )
                -
                CAST(
                    substr(c.cohort_month, 1, 4)
                    AS INTEGER
                )
            ) * 12

            +

            (
                CAST(
                    substr(o.order_month, 6, 2)
                    AS INTEGER
                )
                -
                CAST(
                    substr(c.cohort_month, 6, 2)
                    AS INTEGER
                )
            )
        ) AS month_number

    FROM customer_cohort c

    JOIN customer_orders o
        ON c.customer_id = o.customer_id
),

cohort_size AS (

    SELECT
        cohort_month,
        COUNT(*) AS total_customers

    FROM customer_cohort

    GROUP BY
        cohort_month
)

SELECT
    a.cohort_month,
    a.month_number,

    COUNT(
        DISTINCT a.customer_id
    ) AS active_customers,

    c.total_customers,

    ROUND(
        COUNT(
            DISTINCT a.customer_id
        ) * 100.0
        / c.total_customers,
        2
    ) AS retention_rate

FROM cohort_activity a

JOIN cohort_size c
    ON a.cohort_month = c.cohort_month

WHERE
    a.month_number BETWEEN 0 AND 3

GROUP BY
    a.cohort_month,
    a.month_number,
    c.total_customers

ORDER BY
    a.cohort_month,
    a.month_number
"""

result15 = pd.read_sql(query15, conn)

display(result15)

,cohort_month,month_number,active_customers,total_customers,retention_rate
0,2025-01,0,2,48,4.17
1,2025-01,1,1,48,2.08
2,2025-01,2,2,48,4.17
3,2025-01,3,1,48,2.08
4,2025-02,1,1,55,1.82
5,2025-02,3,2,55,3.64
6,2025-03,0,3,53,5.66
7,2025-03,2,3,53,5.66
8,2025-03,3,2,53,3.77
9,2025-04,0,1,35,2.86


### SQL 16 - Frequently Bought Together

Find pairs of products that were purchased together.

Output:
- product_a
- product_b
- times_bought_together

Same-product pairs and duplicate A-B/B-A pairs must be excluded.

In [337]:
query16 = """
WITH order_products AS (

    SELECT DISTINCT
        order_id,
        product_id

    FROM order_items
),

product_pairs AS (

    SELECT
        a.product_id AS product_a,
        b.product_id AS product_b,

        COUNT(*) AS times_bought_together

    FROM order_products a

    JOIN order_products b

        ON a.order_id = b.order_id

        AND a.product_id < b.product_id

    GROUP BY
        a.product_id,
        b.product_id
)

SELECT
    p1.product_name AS product_a,
    p2.product_name AS product_b,

    pp.times_bought_together

FROM product_pairs pp

JOIN products p1
    ON pp.product_a = p1.product_id

JOIN products p2
    ON pp.product_b = p2.product_id

ORDER BY
    times_bought_together DESC
"""

result16 = pd.read_sql(query16, conn)

display(result16.head(20))

,product_a,product_b,times_bought_together
0,Smartphone 2,Smartphone 3,15
1,Smartphone 1,Jeans 4,3
2,Smartphone 1,Mixer 11,3
3,Smartphone 1,Smartphone 2,2
4,Smartphone 1,Laptop 5,2
5,Smartphone 1,Laptop 10,2
6,Smartphone 1,Laptop 12,2
7,Smartphone 1,Monitor 10,2
8,Smartphone 1,Tablet 3,2
9,Smartphone 1,Camera 1,2


## Part 4 - Python + SQL Integration

In this phase, Python is used to interact with the SQLite database.

The report supports:
- Daily reports
- Weekly reports
- Monthly reports

The report displays:
- Total orders
- Total revenue
- Unique customers
- Top 3 products
- Previous-period revenue
- Percentage change

In [338]:
# Check SQLite connection

print("Database connection is active.")
print("Database:", DB_PATH)

Database connection is active.
Database: ../output/ecommerce.db


In [339]:
# Check SQLite connection

print("Database connection is active.")
print("Database:", DB_PATH)

Database connection is active.
Database: ../output/ecommerce.db


### 4.1 Previous Period Calculation

The previous period is calculated based on the selected report type.

Example:

Daily:
- Current: 2026-07-15
- Previous: 2026-07-14

Weekly:
- Current: 7 days
- Previous: previous 7 days

Monthly:
- Current: selected month
- Previous: previous month

In [340]:
def get_previous_period(report_type, start_date, end_date):

    start = datetime.strptime(
        start_date,
        "%Y-%m-%d"
    )

    end = datetime.strptime(
        end_date,
        "%Y-%m-%d"
    )

    if report_type == "daily":

        previous_start = start - timedelta(days=1)
        previous_end = previous_start

    elif report_type == "weekly":

        previous_end = start - timedelta(days=1)

        previous_start = (
            previous_end
            - timedelta(days=6)
        )

    elif report_type == "monthly":

        previous_end = start - timedelta(days=1)

        number_of_days = (
            end - start
        ).days + 1

        previous_start = (
            previous_end
            - timedelta(days=number_of_days - 1)
        )

    else:

        return None, None

    return (
        previous_start.strftime("%Y-%m-%d"),
        previous_end.strftime("%Y-%m-%d")
    )

In [341]:
previous_start, previous_end = get_previous_period(
    "weekly",
    "2026-07-20",
    "2026-07-26"
)

print("Previous period:")
print(previous_start)
print(previous_end)

Previous period:
2026-07-13
2026-07-19


### 4.2 Current Period Summary

This query calculates:
- Total orders
- Total revenue
- Unique customers

In [342]:
def get_period_summary(start_date, end_date):

    query = """
    SELECT

        COUNT(DISTINCT o.order_id)
        AS total_orders,

        ROUND(
            COALESCE(
                SUM(
                    oi.quantity
                    * oi.unit_price
                    * (1 - oi.discount_percent / 100)
                ),
                0
            ),
            2
        ) AS total_revenue,

        COUNT(
            DISTINCT o.customer_id
        ) AS unique_customers

    FROM orders o

    LEFT JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE DATE(o.order_date)
    BETWEEN DATE(?) AND DATE(?)
    """

    result = pd.read_sql(
        query,
        conn,
        params=[
            start_date,
            end_date
        ]
    )

    return result.iloc[0]

In [343]:
summary = get_period_summary(
    "2026-07-01",
    "2026-07-31"
)

print("Total Orders:", summary["total_orders"])
print("Total Revenue:", summary["total_revenue"])
print("Unique Customers:", summary["unique_customers"])

Total Orders: 13.0
Total Revenue: 1751620.13
Unique Customers: 13.0


### 4.3 Top 3 Products

Find the three products generating the highest revenue during the selected period.

In [344]:
def get_top_products(start_date, end_date):

    query = """
    SELECT

        p.product_name,

        ROUND(
            SUM(
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_percent / 100)
            ),
            2
        ) AS revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE DATE(o.order_date)
    BETWEEN DATE(?) AND DATE(?)

    GROUP BY
        p.product_id,
        p.product_name

    ORDER BY
        revenue DESC

    LIMIT 3
    """

    result = pd.read_sql(
        query,
        conn,
        params=[
            start_date,
            end_date
        ]
    )

    return result

In [345]:
top_products = get_top_products(
    "2026-07-01",
    "2026-07-31"
)

display(top_products)

,product_name,revenue
0,Database Design 19,192648.10
1,Cookware 15,152808.16
2,Algorithms 20,142090.52


### 4.4 Previous-Period Comparison

Compare current-period revenue with previous-period revenue.

Percentage change:

(current revenue - previous revenue)
/
previous revenue
× 100

In [346]:
def calculate_revenue_change(
    current_revenue,
    previous_revenue
):

    if previous_revenue == 0:

        return None

    percentage_change = (
        (
            current_revenue
            - previous_revenue
        )
        / previous_revenue
    ) * 100

    return percentage_change

### 4.5 Complete Sales Report

This function combines:
- Report type
- Date range
- Current-period summary
- Top 3 products
- Previous-period revenue
- Revenue percentage change

In [347]:
def generate_report(
    report_type,
    start_date,
    end_date
):

    # Validate report type

    if report_type not in [
        "daily",
        "weekly",
        "monthly"
    ]:

        print(
            "Invalid report type."
        )

        print(
            "Use daily, weekly or monthly."
        )

        return


    # Get previous period

    previous_start, previous_end = (
        get_previous_period(
            report_type,
            start_date,
            end_date
        )
    )


    # Current period

    current = get_period_summary(
        start_date,
        end_date
    )


    # Previous period

    previous = get_period_summary(
        previous_start,
        previous_end
    )


    # Top 3 products

    top_products = get_top_products(
        start_date,
        end_date
    )


    # Revenue values

    current_revenue = float(
        current["total_revenue"]
    )

    previous_revenue = float(
        previous["total_revenue"]
    )


    # Percentage change

    revenue_change = calculate_revenue_change(
        current_revenue,
        previous_revenue
    )


    # Display report

    print()
    print("=" * 55)
    print("          E-COMMERCE SALES REPORT")
    print("=" * 55)

    print("Report Type :", report_type)
    print("Start Date  :", start_date)
    print("End Date    :", end_date)

    print()
    print("CURRENT PERIOD")
    print("-" * 55)

    print(
        "Total Orders      :",
        current["total_orders"]
    )

    print(
        "Total Revenue     :",
        current["total_revenue"]
    )

    print(
        "Unique Customers  :",
        current["unique_customers"]
    )

    print()
    print("TOP 3 PRODUCTS")
    print("-" * 55)

    display(top_products)

    print()
    print("PREVIOUS PERIOD")
    print("-" * 55)

    print(
        "Previous Start    :",
        previous_start
    )

    print(
        "Previous End      :",
        previous_end
    )

    print(
        "Previous Revenue  :",
        previous_revenue
    )

    if revenue_change is None:

        print(
            "Revenue Change    : Not Available"
        )

    else:

        print(
            "Revenue Change    :",
            round(revenue_change, 2),
            "%"
        )

    print("=" * 55)

In [348]:
# TEst monthly report
generate_report(
    "monthly",
    "2026-07-01",
    "2026-07-31"
)


          E-COMMERCE SALES REPORT
Report Type : monthly
Start Date  : 2026-07-01
End Date    : 2026-07-31

CURRENT PERIOD
-------------------------------------------------------
Total Orders      : 13.0
Total Revenue     : 1751620.13
Unique Customers  : 13.0

TOP 3 PRODUCTS
-------------------------------------------------------


,product_name,revenue
0,Database Design 19,192648.10
1,Cookware 15,152808.16
2,Algorithms 20,142090.52



PREVIOUS PERIOD
-------------------------------------------------------
Previous Start    : 2026-05-31
Previous End      : 2026-06-30
Previous Revenue  : 934051.29
Revenue Change    : 87.53 %


In [349]:
# Test Weekly Report
generate_report(
    "weekly",
    "2025-10-01",
    "2025-10-01"
)


          E-COMMERCE SALES REPORT
Report Type : weekly
Start Date  : 2025-10-01
End Date    : 2025-10-01

CURRENT PERIOD
-------------------------------------------------------
Total Orders      : 2.0
Total Revenue     : 406211.05
Unique Customers  : 2.0

TOP 3 PRODUCTS
-------------------------------------------------------


,product_name,revenue
0,Sql Handbook 7,121401.32
1,Spark Guide 15,104628.65
2,Dress 16,95563.00



PREVIOUS PERIOD
-------------------------------------------------------
Previous Start    : 2025-09-24
Previous End      : 2025-09-30
Previous Revenue  : 0.0
Revenue Change    : Not Available


In [350]:
# Test Daily Report

generate_report(
    "daily",
    "2025-07-04",
    "2025-07-04"
)


          E-COMMERCE SALES REPORT
Report Type : daily
Start Date  : 2025-07-04
End Date    : 2025-07-04

CURRENT PERIOD
-------------------------------------------------------
Total Orders      : 1.0
Total Revenue     : 23506.37
Unique Customers  : 1.0

TOP 3 PRODUCTS
-------------------------------------------------------


,product_name,revenue
0,Jeans 4,52443.19
1,Smartphone 1,-28936.82



PREVIOUS PERIOD
-------------------------------------------------------
Previous Start    : 2025-07-03
Previous End      : 2025-07-03
Previous Revenue  : 0.0
Revenue Change    : Not Available


### 4.6 Interactive Report

The user can enter:
- daily
- weekly
- monthly

along with the required start and end dates.

In [351]:
report_type = input(
    "Enter report type (daily/weekly/monthly): "
).lower()

start_date = input(
    "Enter start date (YYYY-MM-DD): "
)

end_date = input(
    "Enter end date (YYYY-MM-DD): "
)

generate_report(
    report_type,
    start_date,
    end_date
)


          E-COMMERCE SALES REPORT
Report Type : monthly
Start Date  : 2025-10-01
End Date    : 2025-10-31

CURRENT PERIOD
-------------------------------------------------------
Total Orders      : 22.0
Total Revenue     : 3429163.24
Unique Customers  : 19.0

TOP 3 PRODUCTS
-------------------------------------------------------


,product_name,revenue
0,Cloud Computing 5,155744.14
1,Database Design 18,153229.22
2,Database Design 20,125146.28



PREVIOUS PERIOD
-------------------------------------------------------
Previous Start    : 2025-08-31
Previous End      : 2025-09-30
Previous Revenue  : 3103738.27
Revenue Change    : 10.48 %


## Part 5 - Edge Case Handling

This phase tests the system against common data-quality problems.

Test cases:
1. order_items contains an unknown order_id
2. discount_percent is greater than 100
3. quantity is 0
4. order_date is in the future

In [352]:
# Test 1 - Unknown Order ID
def check_unknown_order_ids(order_items_df, orders_df):

    valid_order_ids = set(
        orders_df["order_id"].dropna().astype(str)
    )

    invalid_items = order_items_df[
        ~order_items_df["order_id"].astype(str).isin(valid_order_ids)
    ].copy()

    return invalid_items

In [353]:
unknown_orders = check_unknown_order_ids(
    order_items_cleaned,
    orders_cleaned
)

print("Unknown order_id records:", len(unknown_orders))

display(unknown_orders.head())

Unknown order_id records: 20


,item_id,order_id,product_id,quantity,unit_price,discount_percent
2980,OI002981,O99902981,P0496,3,24728.14,3.77
2981,OI002982,O99902982,P0280,2,33153.40,9.22
2982,OI002983,O99902983,P0389,5,23976.54,3.31
2983,OI002984,O99902984,P0346,4,16170.10,18.14
2984,OI002985,O99902985,P0366,2,6721.85,17.24


In [354]:
# Test 2 - Discount Greater Than 100%
def check_invalid_discounts(order_items_df):

    invalid_discounts = order_items_df[
        pd.to_numeric(
            order_items_df["discount_percent"],
            errors="coerce"
        ) > 100
    ].copy()

    return invalid_discounts

In [355]:
invalid_discounts = check_invalid_discounts(
    order_items_cleaned
)

print(
    "Discount > 100% records:",
    len(invalid_discounts)
)

display(invalid_discounts.head())

Discount > 100% records: 20


,item_id,order_id,product_id,quantity,unit_price,discount_percent
140,OI000141,O00770,P0358,5,9263.79,132.11
141,OI000142,O00378,P0061,4,11473.44,137.05
142,OI000143,O00249,P0491,1,35349.60,118.71
143,OI000144,O00724,P0030,2,26318.70,123.32
144,OI000145,O00698,P0152,3,32307.18,129.34


In [356]:
# Test 3 - Zero Quantity
def check_zero_quantity(order_items_df):

    quantity = pd.to_numeric(
        order_items_df["quantity"],
        errors="coerce"
    )

    zero_quantity = order_items_df[
        quantity == 0
    ].copy()

    return zero_quantity

In [357]:
zero_quantity = check_zero_quantity(
    order_items_cleaned
)

print(
    "Quantity = 0 records:",
    len(zero_quantity)
)

display(zero_quantity.head())

Quantity = 0 records: 20


,item_id,order_id,product_id,quantity,unit_price,discount_percent
120,OI000121,O00258,P0055,0,24927.93,4.25
121,OI000122,O00270,P0130,0,16883.47,17.78
122,OI000123,O00875,P0211,0,13558.52,2.34
123,OI000124,O00803,P0480,0,16221.32,21.65
124,OI000125,O00311,P0436,0,7917.47,7.47


In [358]:
# Test 4 - future Order Dates
def check_future_order_dates(orders_df):

    order_dates = pd.to_datetime(
        orders_df["order_date"],
        errors="coerce"
    )

    today = pd.Timestamp.today().normalize()

    future_orders = orders_df[
        order_dates > today
    ].copy()

    return future_orders

In [359]:
future_orders = check_future_order_dates(
    orders_cleaned
)

print(
    "Future order_date records:",
    len(future_orders)
)

display(future_orders.head())

Future order_date records: 53


,order_id,customer_id,order_date,status,region_code
0,O00001,NaN,2026-09-05 12:35:18,DELIVERED,WEST
14,O00015,NaN,2026-11-04 12:05:12,DELIVERED,CENTRAL
26,O00027,NaN,2026-10-05 18:30:28,CANCELLED,SOUTH
73,O00074,CUST0334,2026-11-06 02:59:53,RETURNED,SOUTH
85,O00086,CUST0396,2026-09-02 01:07:19,DELIVERED,CENTRAL
